In [1]:
# 02_audit_baf — Cell 1: load BAF Base, shape, dtypes, memory, label prevalence
import sys; sys.path.insert(0, "..")
import pandas as pd
import numpy as np
from src.config import PATHS, SEED, Timer

BAF_CSV = PATHS["raw"] / "baf" / "Base.csv"
with Timer("read Base.csv"):
    baf = pd.read_csv(BAF_CSV)

print("shape :", baf.shape)
print("memory: %.1f MB" % (baf.memory_usage(deep=True).sum() / 1e6))
print()
print("dtype counts:\n", baf.dtypes.value_counts(), sep="")
print()

# Text columns = categorical features that will need encoding later
print("categorical (object) columns:")
for c in baf.select_dtypes("object").columns:
    vals = sorted(baf[c].unique())
    print(f"  {c:18s} n_unique={len(vals):2d}  {vals}")
print()

print("fraud_bool counts:\n", baf["fraud_bool"].value_counts(), sep="")
print("prevalence = %.4f" % baf["fraud_bool"].mean())

[read Base.csv] wall = 2.4 s | RSS now = 0.39 GB | peak RSS = 1.00 GB
shape : (1000000, 32)
memory: 520.7 MB

dtype counts:
int64      18
float64     9
object      5
Name: count, dtype: int64

categorical (object) columns:
  payment_type       n_unique= 5  ['AA', 'AB', 'AC', 'AD', 'AE']
  employment_status  n_unique= 7  ['CA', 'CB', 'CC', 'CD', 'CE', 'CF', 'CG']
  housing_status     n_unique= 7  ['BA', 'BB', 'BC', 'BD', 'BE', 'BF', 'BG']
  source             n_unique= 2  ['INTERNET', 'TELEAPP']
  device_os          n_unique= 5  ['linux', 'macintosh', 'other', 'windows', 'x11']

fraud_bool counts:
fraud_bool
0    988971
1     11029
Name: count, dtype: int64
prevalence = 0.0110


In [2]:
# 02_audit_baf — Cell 2: missing values (NaN + negative codes), verified not assumed
NEG_MISSING = ["prev_address_months_count", "current_address_months_count",
               "bank_months_count", "session_length_in_minutes",
               "device_distinct_emails_8w", "intended_balcon_amount"]   # community list (handoff 5.1)
LEGIT_NEGATIVE = {"credit_risk_score", "velocity_6h"}                   # real negative values, not codes

print("true NaN cells in the whole file:", int(baf.isna().sum().sum()), "\n")

num = baf.select_dtypes("number")
neg = (num < 0).sum()
neg = neg[neg > 0].sort_values(ascending=False)

print(f"{'column':30s} {'n_neg':>8s} {'pct':>7s} {'min':>9s} {'n_negvals':>9s} "
      f"{'fraud%|neg':>10s} {'fraud%|ok':>9s}  code  warn")
for c in neg.index:
    m = baf[c] < 0
    warn = "low_n" if neg[c] < 500 else ""
    print(f"{c:30s} {neg[c]:8,d} {100*m.mean():6.2f}% {baf[c].min():9.2f} "
          f"{baf.loc[m, c].nunique():9d} {100*baf.loc[m,'fraud_bool'].mean():9.2f}% "
          f"{100*baf.loc[~m,'fraud_bool'].mean():8.2f}%  {str(c in NEG_MISSING):5s} {warn}")

# Show the actual negative values: exposes the encoding directly
print("\nnegative values by detected column:")
for c in neg.index:
    vals = sorted(baf.loc[baf[c] < 0, c].unique())
    print(f"  {c}: {vals[:20]}{' ...' if len(vals) > 20 else ''}")

detected, expected = set(neg.index), set(NEG_MISSING)
print("\nlisted as code but no negatives found:", expected - detected or "none")
print("negatives outside the code list      :", detected - expected or "none")
assert expected <= detected, "a listed missing-code column has no negatives"
assert detected - expected <= LEGIT_NEGATIVE, "unexpected negative column: inspect before continuing"

# Sanity check only (no automatic reclassification)
for c in LEGIT_NEGATIVE & detected:
    if baf.loc[baf[c] < 0, c].nunique() == 1:
        print(f"WARN: {c} is LEGIT_NEGATIVE but has only 1 distinct negative value; inspect manually")
print("\nASSERTS PASSED")

true NaN cells in the whole file: 0 

column                            n_neg     pct       min n_negvals fraud%|neg fraud%|ok  code  warn
intended_balcon_amount          742,523  74.25%    -15.53    737834      1.31%     0.50%  True  
prev_address_months_count       712,920  71.29%     -1.00         1      1.42%     0.31%  True  
bank_months_count               253,635  25.36%     -1.00         1      1.63%     0.92%  True  
credit_risk_score                14,445   1.44%   -170.00       164      0.39%     1.11%  False 
current_address_months_count      4,254   0.43%     -1.00         1      0.33%     1.11%  True  
session_length_in_minutes         2,015   0.20%     -1.00         1      0.89%     1.10%  True  
device_distinct_emails_8w           359   0.04%     -1.00         1      1.11%     1.10%  True  low_n
velocity_6h                          44   0.00%   -170.60        44      0.00%     1.10%  False low_n

negative values by detected column:
  intended_balcon_amount: [np.float64(

In [3]:
# 02_audit_baf — Cell 3: constants, duplicates, fraud rate and missing-code rate by month
assert baf["month"].dtype.kind in "iu", f"month dtype is {baf['month'].dtype}, split map assumes integer"
assert baf["month"].notna().all(), "month contains missing values"
assert set(baf["month"].unique()) <= set(range(8)), f"unexpected months: {sorted(baf['month'].unique())}"

const = [c for c in baf.columns if baf[c].nunique() <= 1]
print("constant columns:", const)
print("duplicate rows  :", int(baf.duplicated().sum()), "\n")

by_month = baf.groupby("month")["fraud_bool"].agg(n="size", n_fraud="sum", fraud_rate="mean")
by_month["fraud_rate"] = (100 * by_month["fraud_rate"]).round(3)
by_month["split"] = by_month.index.map(lambda m: "train" if m <= 4 else ("valid" if m == 5 else "test"))
print(by_month.to_string(), "\n")

by_split = (by_month.groupby("split")[["n", "n_fraud"]].sum()
              .assign(fraud_rate=lambda d: (100 * d.n_fraud / d.n).round(3)))
print(by_split.to_string())
assert set(by_split.index) == {"train", "valid", "test"}, "a split is empty"

# Exploratory: share of missing-code (negative) values per month (feature-distribution shift indicator)
neg_by_month = baf.groupby("month")[NEG_MISSING].apply(lambda g: (g < 0).mean() * 100).round(3)
print("\nnegative-code rate by month (%):")
print(neg_by_month.T.to_string())

constant columns: ['device_fraud_count']
duplicate rows  : 0 

            n  n_fraud  fraud_rate  split
month                                    
0      132440     1500       1.133  train
1      127620     1198       0.939  train
2      136979     1198       0.875  train
3      150936     1392       0.922  train
4      127691     1452       1.137  train
5      119323     1411       1.183  valid
6      108168     1450       1.341   test
7       96843     1428       1.475   test 

            n  n_fraud  fraud_rate
split                             
test   205011     2878       1.404
train  675666     6740       0.998
valid  119323     1411       1.183

negative-code rate by month (%):
month                              0       1       2       3       4       5       6       7
prev_address_months_count     70.026  75.731  66.236  64.556  72.323  82.562  75.154  65.264
current_address_months_count   0.461   0.269   0.469   0.473   0.406   0.168   0.362   0.862
bank_months_count          

In [4]:
# 02_audit_baf — Cell 4: descriptive evidence for DECISION #5 + velocity_6h anomaly location
v = baf["intended_balcon_amount"]
neg_v, pos_v = v[v < 0], v[v >= 0]
for name, s in [("negatives", neg_v), ("non-neg  ", pos_v)]:
    print(f"{name}: n={len(s):,}  min={s.min():.2f}  p1={s.quantile(.01):.2f}  "
          f"p50={s.median():.2f}  p99={s.quantile(.99):.2f}  max={s.max():.2f}")
print("exact zeros:", int((v == 0).sum()))

# Fraud rate across value bins: flat inside negatives + jump at 0 -> the sign carries the signal;
# smooth trend across all bins -> behaves like a continuous feature
bins = pd.cut(v, [-np.inf, -10, -5, -2, -1, 0, 10, 50, np.inf], right=False)
tab = baf.groupby(bins, observed=True)["fraud_bool"].agg(n="size", fraud_rate="mean")
tab["fraud_rate"] = (100 * tab["fraud_rate"]).round(2)
print("\n", tab.to_string())

vel_neg = baf["velocity_6h"] < 0
print("\nvelocity_6h negatives by month:", baf.loc[vel_neg, "month"].value_counts().sort_index().to_dict())

negatives: n=742,523  min=-15.53  p1=-1.87  p50=-1.01  p99=-0.18  max=-0.00
non-neg  : n=257,477  min=0.00  p1=1.53  p50=32.43  p99=105.71  max=112.96
exact zeros: 0

                              n  fraud_rate
intended_balcon_amount                    
[-inf, -10.0)               85        0.00
[-10.0, -5.0)              746        0.27
[-5.0, -2.0)              2367        0.51
[-2.0, -1.0)            376404        1.21
[-1.0, 0.0)             362921        1.42
[0.0, 10.0)              19657        0.46
[10.0, 50.0)            180597        0.47
[50.0, inf)              57223        0.60

velocity_6h negatives by month: {1: 1, 2: 1, 3: 2, 4: 5, 5: 11, 6: 14, 7: 10}


In [5]:
# 02_audit_baf — Cell 5: minimal cleaning, Parquet export, round-trip schema checks on both copies
clean = baf.drop(columns=["device_fraud_count"])        # constant in Base: zero information
CAT_COLS = ["payment_type", "employment_status", "housing_status", "source", "device_os"]
for c in CAT_COLS:
    clean[c] = clean[c].astype("category")
clean["month"] = clean["month"].astype("int8")
clean["fraud_bool"] = clean["fraud_bool"].astype("int8")

OUTS = [PATHS["lake"] / "baf_base_clean.parquet",       # archive copy (HDD)
        PATHS["work"] / "baf_base_clean.parquet"]       # working copy (SSD)
with Timer("write parquet x2"):
    for p in OUTS:
        clean.to_parquet(p, index=False)

for p in OUTS:
    chk = pd.read_parquet(p)
    assert chk.shape == (len(baf), baf.shape[1] - 1), chk.shape
    assert chk.columns.tolist() == clean.columns.tolist()
    assert chk.dtypes.equals(clean.dtypes)
    assert int(chk["fraud_bool"].sum()) == int(baf["fraud_bool"].sum())
    for c in CAT_COLS:
        assert list(chk[c].cat.categories) == list(clean[c].cat.categories), c
    print(f"OK  {p}  {p.stat().st_size / 1e6:.1f} MB")
print("RAM of clean frame: %.1f MB" % (clean.memory_usage(deep=True).sum() / 1e6))

[write parquet x2] wall = 2.3 s | RSS now = 0.90 GB | peak RSS = 1.00 GB
OK  D:\m1\lake\baf_base_clean.parquet  69.5 MB
OK  C:\m1\work\baf_base_clean.parquet  69.5 MB
RAM of clean frame: 199.0 MB


In [6]:
# 02_audit_baf — Cell 6: machine-readable audit summary for the day-7 report
import json
split = np.where(baf["month"] <= 4, "train", np.where(baf["month"] == 5, "valid", "test"))
check_cols = NEG_MISSING + sorted(LEGIT_NEGATIVE)
audit = {
    "rows": len(baf), "cols_raw": baf.shape[1], "fraud_total": int(baf["fraud_bool"].sum()),
    "prevalence_by_split": baf.groupby(split)["fraud_bool"].mean().round(5).to_dict(),
    "negative_rate": {c: round(float((baf[c] < 0).mean()), 5) for c in check_cols},
    "fraud_rate_neg_vs_nonneg": {c: [round(float(baf.loc[baf[c] < 0, "fraud_bool"].mean()), 5),
                                     round(float(baf.loc[baf[c] >= 0, "fraud_bool"].mean()), 5)]
                                 for c in check_cols},
    "constant_cols_dropped": ["device_fraud_count"],
    "duplicates": int(baf.duplicated().sum()),
}
out = PATHS["repo"] / "reports" / "audit_baf.json"
out.write_text(json.dumps(audit, indent=2))
print(json.dumps(audit, indent=2))

{
  "rows": 1000000,
  "cols_raw": 32,
  "fraud_total": 11029,
  "prevalence_by_split": {
    "test": 0.01404,
    "train": 0.00998,
    "valid": 0.01183
  },
  "negative_rate": {
    "prev_address_months_count": 0.71292,
    "current_address_months_count": 0.00425,
    "bank_months_count": 0.25363,
    "session_length_in_minutes": 0.00201,
    "device_distinct_emails_8w": 0.00036,
    "intended_balcon_amount": 0.74252,
    "credit_risk_score": 0.01444,
    "velocity_6h": 4e-05
  },
  "fraud_rate_neg_vs_nonneg": {
    "prev_address_months_count": [
      0.01421,
      0.00312
    ],
    "current_address_months_count": [
      0.00329,
      0.01106
    ],
    "bank_months_count": [
      0.01632,
      0.00923
    ],
    "session_length_in_minutes": [
      0.00893,
      0.01103
    ],
    "device_distinct_emails_8w": [
      0.01114,
      0.01103
    ],
    "intended_balcon_amount": [
      0.01313,
      0.00498
    ],
    "credit_risk_score": [
      0.00388,
      0.01113
    ],